# 🚀 QuantAgent — GPU Training on Colab

This notebook trains the RL trading model on a **free T4 GPU** (~30 min).

**How to use:**
1. Click **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. Wait ~30 minutes
4. Download the model file when prompted
5. Upload it to your QuantAgent dashboard (RL Brain → Upload Model)

In [ ]:
# Cell 1: Install dependencies
!pip install -q stable-baselines3 sb3-contrib torch yfinance pandas numpy ta gymnasium

In [ ]:
# Cell 2: Configuration
TICKER = 'MSFT' #@param {type:"string"}
TIMEFRAME = '1d' #@param ["1d", "1h", "15m"]
TOTAL_STEPS = 500000 #@param {type:"integer"}
PERIOD = '5y'            # How much historical data to use

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta

print(f'Fetching {PERIOD} of {TICKER} data...')
df = yf.download(TICKER, period=PERIOD, interval=TIMEFRAME, auto_adjust=True)
print(f'Got {len(df)} bars')

# Build 45-dim feature vector (same as QuantAgent feature_pipeline)
def build_features(df):
    f = pd.DataFrame(index=df.index)
    # Ensure 1-dimensional Series are passed to ta functions
    c = df['Close'].squeeze()
    h = df['High'].squeeze()
    l = df['Low'].squeeze()
    v = df['Volume'].squeeze()

    # Returns & volatility (dims 0-9)
    for w in [5, 10, 20, 60]:
        f[f'ret_{w}'] = c.pct_change(w)
    f['log_ret'] = np.log(c / c.shift(1))
    for w in [10, 20, 60]:
        f[f'vol_{w}'] = f['log_ret'].rolling(w).std()
    f['vol_ratio'] = f['vol_10'] / (f['vol_60'] + 1e-8)
    f['ret_vol_ratio'] = f['ret_5'] / (f['vol_20'] + 1e-8)

    # Technical indicators (dims 10-19)
    f['rsi_14'] = ta.momentum.RSIIndicator(c, 14).rsi() / 100
    macd = ta.trend.MACD(c)
    f['macd'] = macd.macd_diff()
    f['macd_signal'] = macd.macd_signal()
    bb = ta.volatility.BollingerBands(c, 20)
    f['bb_pct'] = (c - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband() + 1e-8)
    f['atr_14'] = ta.volatility.AverageTrueRange(h, l, c, 14).average_true_range() / c
    stoch = ta.momentum.StochasticOscillator(h, l, c, 14, 3)
    f['stoch_k'] = stoch.stoch() / 100
    f['stoch_d'] = stoch.stoch_signal() / 100
    f['williams_r'] = ta.momentum.WilliamsRIndicator(h, l, c, 14).williams_r() / 100
    f['roc_10'] = ta.momentum.ROCIndicator(c, 10).roc() / 100
    f['cci_20'] = ta.trend.CCIIndicator(h, l, c, 20).cci() / 200

    # Volume features (dims 20-24)
    f['vol_sma_ratio'] = v / (v.rolling(20).mean() + 1)
    f['obv_norm'] = ta.volume.OnBalanceVolumeIndicator(c, v).on_balance_volume()
    f['obv_norm'] = (f['obv_norm'] - f['obv_norm'].rolling(60).mean()) / (f['obv_norm'].rolling(60).std() + 1e-8)
    f['vwap_dev'] = 0.0  # simplified
    f['volume_trend'] = v.rolling(5).mean() / (v.rolling(20).mean() + 1)
    f['dollar_volume_rank'] = (c * v).rolling(20).mean().rank(pct=True)

    # Trend features (dims 25-28)
    for w in [10, 20, 50, 200]:
        f[f'sma_dist_{w}'] = (c - c.rolling(w).mean()) / (c.rolling(w).std() + 1e-8)

    # Regime placeholder (dims 29-31) - will be random since HMM not available
    f['regime_trending'] = 0.5
    f['regime_mean_rev'] = 0.3
    f['regime_high_vol'] = 0.2

    # Extra features to reach 45 dims (dims 32-44)
    f['hurst'] = 0.5
    f['adx'] = ta.trend.ADXIndicator(h, l, c, 14).adx() / 100
    f['di_plus'] = ta.trend.ADXIndicator(h, l, c, 14).adx_pos() / 100
    f['di_minus'] = ta.trend.ADXIndicator(h, l, c, 14).adx_neg() / 100
    f['mfi'] = ta.volume.MFIIndicator(h, l, c, v, 14).money_flow_index() / 100
    f['cmf'] = ta.volume.ChaikinMoneyFlowIndicator(h, l, c, v, 20).chaikin_money_flow()
    for w in [5, 10, 20]:
        f[f'autocorr_{w}'] = f['log_ret'].rolling(w).apply(lambda x: x.autocorr() if len(x) > 2 else 0, raw=False)
    f['skew_20'] = f['log_ret'].rolling(20).skew()
    f['kurt_20'] = f['log_ret'].rolling(20).kurt()
    f['max_dd_60'] = c.rolling(60).apply(lambda x: (x[-1] - x.max()) / (x.max() + 1e-8), raw=True)
    f['up_capture'] = f['ret_5'].clip(lower=0).rolling(20).mean() / (f['vol_20'] + 1e-8)

    # Ensure exactly 45 columns
    f = f.iloc[:, :45] if f.shape[1] > 45 else f
    while f.shape[1] < 45:
        f[f'pad_{f.shape[1]}'] = 0.0

    f = f.fillna(0).replace([np.inf, -np.inf], 0)
    return f

features = build_features(df)
print(f'Features shape: {features.shape}')

In [ ]:
# Cell 4: Build Gym environment
import gymnasium as gym
from gymnasium import spaces

class TradingEnv(gym.Env):
    """Simple trading environment for RL training."""
    def __init__(self, df, features_df, initial_capital=100_000):
        super().__init__()
        self.df = df
        self.features = features_df.values.astype(np.float32)
        self.closes = df['Close'].values.astype(np.float64)
        self.initial_capital = initial_capital
        self.n_features = self.features.shape[1]

        # Action: [direction (-1 to 1), size (0 to 1)]
        self.action_space = spaces.Box(low=np.array([-1, 0]), high=np.array([1, 1]), dtype=np.float32)
        # Observation: features + position + pnl
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.n_features,), dtype=np.float32
        )
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Start from a random point (burn-in first 252 bars)
        max_start = len(self.df) - 252
        self.start_idx = np.random.randint(252, max(253, max_start))
        self.current_step = 0
        self.capital = self.initial_capital
        self.position = 0.0
        self.peak_capital = self.initial_capital
        self.episode_returns = []
        return self._get_obs(), {}

    def _get_obs(self):
        idx = self.start_idx + self.current_step
        if idx >= len(self.features):
            idx = len(self.features) - 1
        obs = self.features[idx].copy()
        obs = np.where(np.isfinite(obs), obs, 0.0)
        return obs

    def step(self, action):
        idx = self.start_idx + self.current_step
        if idx >= len(self.closes) - 1:
            return self._get_obs(), 0.0, True, False, {}

        direction = float(np.clip(action[0], -1, 1))
        size_mod = float(np.clip(action[1], 0, 1))
        target_pos = direction * size_mod * 0.50

        # Transaction cost
        delta = abs(target_pos - self.position)
        cost = delta * self.capital * 0.002
        self.capital -= cost
        self.position = target_pos

        # PnL
        price_ret = (self.closes[idx + 1] - self.closes[idx]) / (self.closes[idx] + 1e-8)
        pnl = self.position * price_ret
        self.capital *= (1 + pnl)
        self.episode_returns.append(pnl)

        # Drawdown
        self.peak_capital = max(self.peak_capital, self.capital)
        drawdown = (self.peak_capital - self.capital) / (self.peak_capital + 1e-8)

        # Reward: risk-adjusted return with drawdown penalty
        down_rets = [r for r in self.episode_returns if r < 0]
        down_std = np.std(down_rets) if len(down_rets) >= 2 else 0.01
        sortino_proxy = pnl / (down_std + 1e-6)

        reward = sortino_proxy * 0.5 - cost * 100 - drawdown * 2.0

        self.current_step += 1
        done = (self.current_step >= 200) or (idx + 1 >= len(self.closes) - 1) or (drawdown > 0.20)

        return self._get_obs(), float(reward), done, False, {'capital': self.capital, 'drawdown': drawdown}

# Create environment
env = TradingEnv(df, features)
print(f'Environment created: obs_dim={env.observation_space.shape}, action_dim={env.action_space.shape}')

In [ ]:
# Cell 5: Train with RecurrentPPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import DummyVecEnv
import time

def make_env():
    return TradingEnv(df, features)

vec_env = DummyVecEnv([make_env for _ in range(4)])  # 4 parallel envs

model = RecurrentPPO(
    'MlpLstmPolicy',
    vec_env,
    learning_rate=3e-4,
    n_steps=256,
    batch_size=64,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,
    verbose=1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

print(f'Training {TOTAL_STEPS:,} steps on {model.device}...')
start = time.time()
model.learn(total_timesteps=TOTAL_STEPS, progress_bar=True)
elapsed = time.time() - start
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes!')

In [ ]:
# Cell 6: Save & download model
import os

model_name = f'rl_{TICKER}_{TIMEFRAME}'
model.save(model_name)
print(f'Model saved: {model_name}.zip ({os.path.getsize(model_name + ".zip") / 1024:.1f} KB)')

# Auto-download
from google.colab import files
files.download(f'{model_name}.zip')
print('\n📥 Download started! Upload this file to QuantAgent → RL Brain → Upload Model')